# Wikipedia pageviews (Lane B: the agent's output, captured)

**SISMID 2026 - Day 2, 9:00 - Data beyond Google Trends.**

> Every cell below is a captured example of what a coding agent (Codex, Claude Code, or
> Antigravity CLI) produces from the prompts in the **Lane A** notebook. Run it top to bottom
> as a backup, or read it to see what good output looks like.

Wikipedia pageviews are a free, hourly, **multi-language** access signal with an official
API and **no key**. Unlike Google Trends they are **stable and reproducible** - a good
contrast. Target: monthly views of the *Dengue* article in English, Spanish, Portuguese.


## Step 0: the helper (from the Step 0 prompt)


In [ ]:
# --- Produced by the agent from the Plan A / Step 0 prompt ---
import urllib.request, json, os
import pandas as pd, matplotlib.pyplot as plt

USER_AGENT = 'SISMID2026-course/1.0 (your-email@example.com)'   # put a real contact
CACHE_PATHS = ['../data/wikipedia_dengue_pageviews_cached.csv',
               'data/wikipedia_dengue_pageviews_cached.csv',
               './wikipedia_dengue_pageviews_cached.csv']

def wiki_fetch(article, wiki, start='2016010100', end='2025120100'):
    """Monthly Wikipedia pageviews for one article. Returns DataFrame(date, views) or None."""
    url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
           f'{wiki}/all-access/all-agents/{article}/monthly/{start}/{end}')
    try:
        req = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
        items = json.loads(urllib.request.urlopen(req, timeout=30).read())['items']
        return pd.DataFrame([{'date': pd.to_datetime(it['timestamp'][:8], format='%Y%m%d'),
                              'views': it['views']} for it in items])
    except Exception as e:
        print(f'Wikipedia live pull failed ({type(e).__name__}): {e}')
        return None

def load_cache():
    for p in CACHE_PATHS:
        if os.path.exists(p):
            print(f'Using cached snapshot: {p}')
            return pd.read_csv(p, parse_dates=['date'])
    raise FileNotFoundError('Wikipedia cache not found; check the data/ folder.')

def get_dengue_wiki():
    en = wiki_fetch('Dengue_fever', 'en.wikipedia')
    es = wiki_fetch('Dengue', 'es.wikipedia')
    pt = wiki_fetch('Dengue', 'pt.wikipedia')
    if en is None or es is None or pt is None:
        return load_cache()   # any failure -> the merged cached snapshot
    return (en.rename(columns={'views':'dengue_en'})
              .merge(es.rename(columns={'views':'dengue_es'}), on='date')
              .merge(pt.rename(columns={'views':'dengue_pt'}), on='date'))


## Step 1: pull three languages and overlay

The same concept in three languages gives three signals. Spanish speaks for much of
Latin America, Portuguese for Brazil, English for global attention.


In [ ]:
df = get_dengue_wiki()
print('rows:', len(df), '| range:', df['date'].min().date(), 'to', df['date'].max().date())
cols = [c for c in df.columns if c != 'date']
plt.figure(figsize=(10,4))
for c in cols: plt.plot(df['date'], df[c], label=c)
plt.legend(); plt.title('Wikipedia pageviews: Dengue by language'); plt.ylabel('views')
plt.xlabel('date'); plt.tight_layout(); plt.show()
df.tail()


## Step 2: reproducibility (the contrast with Google Trends)

Pull the same series twice. A stable API returns **identical** numbers - the opposite of
Google Trends' sampling noise. That is exactly why Wikipedia is a useful complement.


In [ ]:
a = wiki_fetch('Dengue_fever', 'en.wikipedia')
b = wiki_fetch('Dengue_fever', 'en.wikipedia')
if a is not None and b is not None:
    print('identical pulls?', a['views'].equals(b['views']))
else:
    print('live unavailable; from the cache this series is fixed anyway.')


## Step 3: sanity-check

Range, gaps, and where the languages agree or diverge (divergence is signal: local
outbreaks show up in the local-language article first).


In [ ]:
print('missing per column:'); print(df.isna().sum())
print('\ncorrelation between languages:'); print(df[cols].corr().round(2))


## Step 4: save the tidy CSV

One date column, one column per language. This is a stream for your capstone bundle.


In [ ]:
df.to_csv('dengue_wikipedia.csv', index=False)
print('saved dengue_wikipedia.csv', df.shape)


## Reflection

- Wikipedia is the **well-behaved** stream: public API, no key, no datacenter block,
  reproducible. The five-step loop is identical to Google Trends; only the fetch differs.
- **Language is a feature**: non-English articles reach populations Google Trends misses.
- **Stretch:** swap in your own disease/article and languages, or align to weekly and
  overlay against the Google Trends series you built on Day 1.
